In [90]:
from bayes_opt import BayesianOptimization
from sklearn.ensemble import RandomForestRegressor
import os
import numpy as np
import pandas as pd

In [91]:
processed_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
input_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "InputData.csv")
output_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "OutputData.csv")

df_input = pd.read_csv(input_data_path)
df_output = pd.read_csv(output_data_path)

In [92]:
input_columns = ["Flow_C", "Flow_PP", "Flow_P", "Current_P"]
output_column_removal = ["Removal"]
output_column_recovery = ["Recovery"]

X = df_input[input_columns].values
y_removal = df_output[output_column_removal].values
y_recovery = df_output[output_column_recovery].values

In [93]:
model_removel = RandomForestRegressor()
model_removel.fit(X, y_removal)
model_recovery = RandomForestRegressor()
model_recovery.fit(X, y_recovery)

def black_box_removal(**kwargs):
    x1 = kwargs['Flow_C']
    x2 = kwargs['Flow_PP']
    x3 = kwargs['Flow_P']
    x4 = kwargs['Current_P']
    return model_removel.predict([[x1, x2, x3, x4]])[0]
def black_box_recovery(**kwargs):
    x1 = kwargs['Flow_C']
    x2 = kwargs['Flow_PP']
    x3 = kwargs['Flow_P']
    x4 = kwargs['Current_P']
    return model_recovery.predict([[x1, x2, x3, x4]])[0]

c:\Users\david\AppData\Local\pypoetry\Cache\virtualenvs\capbioco2-optimization-LdjI1cg_-py3.13\Lib\site-packages\sklearn\base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\david\AppData\Local\pypoetry\Cache\virtualenvs\capbioco2-optimization-LdjI1cg_-py3.13\Lib\site-packages\sklearn\base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [94]:
pbounds = {
    'Flow_C': (min(df_input['Flow_C']), max(df_input['Flow_C'])),
    'Flow_PP': (min(df_input['Flow_PP']), max(df_input['Flow_PP'])),
    'Flow_P': (min(df_input['Flow_P']), max(df_input['Flow_P'])),
    'Current_P': (min(df_input['Current_P']), max(df_input['Current_P']))
}


In [95]:
optimizer_removal = BayesianOptimization(
    f=black_box_removal,
    pbounds=pbounds,
    random_state=42,
    verbose=2,
    allow_duplicate_points=True
)
optimizer_recovery = BayesianOptimization(
    f=black_box_recovery,
    pbounds=pbounds,
    random_state=42,
    verbose=2,
    allow_duplicate_points=True
)

for i in range(len(df_input)):
    optimizer_removal.register(
        params={
            'Flow_C': df_input.iloc[i]['Flow_C'],
            'Flow_PP': df_input.iloc[i]['Flow_PP'],
            'Flow_P': df_input.iloc[i]['Flow_P'],
            'Current_P': df_input.iloc[i]['Current_P']
        },
        target=df_output.iloc[i]['Removal']
    )
    optimizer_recovery.register(
        params={
            'Flow_C': df_input.iloc[i]['Flow_C'],
            'Flow_PP': df_input.iloc[i]['Flow_PP'],
            'Flow_P': df_input.iloc[i]['Flow_P'],
            'Current_P': df_input.iloc[i]['Current_P']
        },
        target=df_output.iloc[i]['Recovery']
    )

optimizer_removal.maximize(
    init_points=0,
    n_iter=20
)
optimizer_recovery.maximize(
    init_points=0,
    n_iter=20
)


| 2         | 0.96      | 0.1923076 | 0.2307692 | 0.2307692 | 1.0       |
| 2         | 0.36      | 0.1923076 | 0.2307692 | 0.2307692 | 1.0       |
| 3         | 0.95      | 0.1153846 | 0.1538461 | 0.1538461 | 1.0       |
| 3         | 0.36      | 0.1153846 | 0.1538461 | 0.1538461 | 1.0       |
| 4         | 0.89      | 0.0769230 | 0.0769230 | 0.0769230 | 1.0       |
| 4         | 0.34      | 0.0769230 | 0.0769230 | 0.0769230 | 1.0       |
| 5         | 0.83      | 0.0384615 | 0.0384615 | 0.0384615 | 1.0       |
| 5         | 0.33      | 0.0384615 | 0.0384615 | 0.0384615 | 1.0       |
| 6         | 0.97      | 0.2692307 | 0.3076923 | 0.3076923 | 1.0       |
| 6         | 0.36      | 0.2692307 | 0.3076923 | 0.3076923 | 1.0       |
Data point [0.19230769 0.23076923 0.23076923 1.        ] is not unique. 1 duplicates registered. Continuing ...
| 7         | 0.98      | 0.1923076 | 0.2307692 | 0.2307692 | 1.0       |
Data point [0.19230769 0.23076923 0.23076923 1.        ] is not unique. 1 

In [97]:
def denormalize(norm_values, bounds):
    return {
        k: norm_values[k] * (bounds[k][1] - bounds[k][0]) + bounds[k][0]
        for k in norm_values
    }

input_bounds = {
    'Flow_C': (0.2, 1.5),
    'Flow_PP': (0.2, 1.5),
    'Flow_P': (0.2, 1.5),
    'Current_P': (35, 55)
}

output_bounds = {
    'Removal': (0, 100),
    'Recovery': (0, 100),
}

print("------------------------Removal:")
print(denormalize(optimizer_removal.max.get('params'), input_bounds))
print(denormalize({'Removal': optimizer_removal.max.get('target')}, output_bounds))

print("------------------------Recovery:")
print(denormalize(optimizer_recovery.max.get('params'), input_bounds))
print(denormalize({'Recovery': optimizer_recovery.max.get('target')}, output_bounds))

------------------------Removal:
{'Flow_C': np.float64(0.75), 'Flow_PP': np.float64(0.8), 'Flow_P': np.float64(0.8), 'Current_P': np.float64(55.0)}
{'Removal': np.float64(99.0)}
------------------------Recovery:
{'Flow_C': np.float64(0.45), 'Flow_PP': np.float64(0.49999999999999994), 'Flow_P': np.float64(0.49999999999999994), 'Current_P': np.float64(55.0)}
{'Recovery': np.float64(36.0)}


# nach jeder iteration einmal removal/recovery und fehler ausgeben als graph, sollte laut nick stagnieren/einschwingen, gut um zwischen verschiednen Modellvarianten zu vergleichen